# 04. Treinamento e Comparação de Modelos

## Contexto e Estratégia de Validação
Nesta etapa, treinamos e avaliamos múltiplos algoritmos de aprendizado de máquina supervisionado para determinar quais arquiteturas apresentam melhor desempenho no conjunto de dados.

### Conceitos Chave:
1. **Validação Cruzada Estratificada (Stratified K-Fold):** Dividimos os dados de treino em $k=5$ dobras (*folds*). A cada iteração, 4 dobras são usadas para treino e 1 para validação, garantindo que todas as amostras passem por validação sem vazamento de dados do conjunto de teste final.
2. **Diversidade de Algoritmos:** Avaliamos 5 famílias distintas de classificadores:
   - **Logistic Regression:** Modelo linear probabilístico, serve como baseline interpretável.
   - **Decision Tree:** Modelo baseado em regras de decisão não lineares.
   - **Random Forest:** Métodos de *ensemble* (bagging) baseados em múltiplas árvores de decisão.
   - **K-Nearest Neighbors (KNN):** Modelo baseado em distância no espaço de atributos.
   - **Support Vector Machine (SVM):** Modelo que busca um hiperplano ótimo de separação.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Carregamento das matrizes pré-processadas no Notebook 03
data = np.load("../data/processed/train_test_data.npz")
X_train, y_train = data['X_train'], data['y_train']

print(f"Dados de treino carregados com sucesso!")
print(f"Dimensão das Features (X_train): {X_train.shape}")
print(f"Dimensão do Target (y_train):   {y_train.shape}")

Dados de treino carregados com sucesso!
Dimensão das Features (X_train): (242, 28)
Dimensão do Target (y_train):   (242,)


## 1. Definição dos Classificadores e Validação Cruzada

Instanciamos os 5 modelos utilizando sementes aleatórias fixas (`random_state=42`) para garantir a reprodutibilidade dos experimentos. Em seguida, configuramos o algoritmo `StratifiedKFold` com $k=5$.

In [2]:
from sklearn.calibration import CalibratedClassifierCV

# Instanciação dos algoritmos baseline (SVM atualizado com CalibratedClassifierCV)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": CalibratedClassifierCV(SVC(random_state=42))
}

# Configuração da Validação Cruzada (5 folds com embaralhamento)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Execução do treinamento e avaliação cruzada
results = {}

print("--- Desempenho na Validação Cruzada (Acurácia Média) ---")
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f"{name:<20}: {scores.mean():.4f} (+/- {scores.std():.4f})")

--- Desempenho na Validação Cruzada (Acurácia Média) ---
Logistic Regression : 0.8471 (+/- 0.0100)
Random Forest       : 0.8098 (+/- 0.0311)
Decision Tree       : 0.6897 (+/- 0.0575)
KNN                 : 0.8264 (+/- 0.0254)
SVM                 : 0.8180 (+/- 0.0253)


## 2. Consolidação e Comparação dos Resultados

Organização dos resultados em um DataFrame do Pandas para facilitar a comparação visual do desempenho médio e do desvio padrão obtido por cada algoritmo.

In [3]:
# Tabela comparativa dos modelos
results_df = pd.DataFrame({
    "Modelo": list(results.keys()),
    "Acurácia Média (CV)": [scores.mean() for scores in results.values()],
    "Desvio Padrão": [scores.std() for scores in results.values()]
}).sort_values(by="Acurácia Média (CV)", ascending=False).reset_index(drop=True)

results_df

,Modelo,Acurácia Média (CV),Desvio Padrão
0,Logistic Regression,0.847109,0.009977
1,KNN,0.826361,0.025366
2,SVM,0.818027,0.025267
3,Random Forest,0.809779,0.031097
4,Decision Tree,0.689711,0.057529


## 3. Síntese do Desempenho dos Modelos

1. **Modelos Lineares e Ensembles:** A Regressão Logística e a Random Forest frequentemente atingem os melhores resultados na validação cruzada devido ao bom comportamento em conjuntos de dados com dimensionalidade moderada.
2. **Ajustes Próximos:** A acurácia isolada na validação cruzada é um bom direcionador inicial, porém a decisão final do modelo será tomada no **Notebook 05**, avaliando métricas cruciais no contexto médico como **Recall**, **Precision** e **ROC AUC** nos dados de teste não vistos.